# Prism Eval — GPT-2 124M on OpenWebText

**The real test.** Does the 13x hold on a production-scale model?

- GPT-2 small (124M params, 12 layers, 768 hidden)
- OpenWebText (~9B tokens)
- 10K steps (short run, enough to see signal)
- Single A100

If Prism Score > 2.0 here, it transfers beyond Shakespeare.
If Prism Score > 5.0, it's a real result for foundation model training.

**Note:** OpenWebText download + preprocessing takes ~2 hours and ~50GB disk.
Run Cell 1 first and let it finish before proceeding.

In [ ]:
# Cell 1: Setup + download OpenWebText (slow — ~2 hours)
import os, shutil, subprocess

os.chdir('/content')
if os.path.exists('/content/nanogpt-prism'):
    shutil.rmtree('/content/nanogpt-prism')
subprocess.run(['git', 'clone',
    'https://github.com/realityinspector/nanogpt-prism.git',
    '/content/nanogpt-prism'], check=True)
os.chdir('/content/nanogpt-prism/src')
subprocess.run(['pip', 'install', '-q', 'transformers', 'tiktoken', 'datasets'])

import torch
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} ({vram:.0f}GB)')

# Check if OpenWebText is already prepared
if os.path.exists('data/openwebtext/train.bin'):
    size_gb = os.path.getsize('data/openwebtext/train.bin') / 1e9
    print(f'OpenWebText already prepared ({size_gb:.1f}GB)')
else:
    print('Preparing OpenWebText (this takes ~2 hours)...')
    print('Downloads ~50GB, preprocesses to ~17GB train.bin')
    !python data/openwebtext/prepare.py
    print('Done.')

In [ ]:
# Cell 2: Train GPT-2 teacher + extract fingerprint
import subprocess, os, re, time, json, gc
os.chdir('/content/nanogpt-prism/src')

TEACHER_STEPS = 2000
cache = '.prism_cache/gpt2_teacher'

if os.path.exists(f'{cache}/directions.pt'):
    print('Teacher fingerprint cached.')
else:
    # Option A: Extract from HuggingFace pretrained GPT-2 (free, instant)
    # This is the "public pretrained model" use case
    print('Extracting fingerprint from pretrained GPT-2 (HuggingFace)...')
    r = subprocess.run([
        'python', 'prism_extract.py', '--hf', 'gpt2', '--out', cache,
    ], capture_output=True, text=True, timeout=300)
    print(r.stdout[-500:] if r.stdout else '')
    if r.returncode != 0:
        print(f'ERROR: {r.stderr[-300:]}')
    else:
        print('Done. Using pretrained GPT-2 as spectral teacher.')
        print('(No training needed — the pretrained model IS the teacher.)')

In [ ]:
# Cell 3: Run baseline (10K steps, GPT-2 config)
import subprocess, time, re, os
os.chdir('/content/nanogpt-prism/src')

STEPS = 10000
EVAL = 500

log_path = '/content/gpt2_baseline.txt'
if os.path.exists(log_path):
    print('Baseline cached. Skipping.')
else:
    print('='*60)
    print('  GPT-2 124M BASELINE (standard init)')
    print('='*60)
    t0 = time.time()
    r = subprocess.run([
        'python', 'train.py', 'config/train_gpt2.py',
        f'--max_iters={STEPS}',
        f'--eval_interval={EVAL}',
        '--eval_iters=50',
        '--log_interval=100',
        '--out_dir=out-gpt2-baseline',
        '--wandb_log=False',
        '--compile=True',
        '--prism_init=False',
        # Single GPU overrides
        '--gradient_accumulation_steps=40',
        '--batch_size=12',
    ], capture_output=True, text=True, timeout=36000)  # 10 hour timeout
    wall = time.time() - t0
    with open(log_path, 'w') as f:
        f.write(r.stdout)
    if r.returncode != 0:
        print(f'ERROR: {r.stderr[-500:]}')
    print(f'Wall time: {wall:.0f}s ({wall/3600:.1f}h)')
    # Show last few evals
    for line in r.stdout.split('\n')[-50:]:
        if 'val loss' in line:
            print(f'  {line.strip()}')

In [ ]:
# Cell 4: Run Prism Recipe (10K steps, GPT-2 config)
import subprocess, time, os
os.chdir('/content/nanogpt-prism/src')

cache = '.prism_cache/gpt2_teacher'
log_path = '/content/gpt2_prism.txt'

if os.path.exists(log_path):
    print('Prism cached. Skipping.')
else:
    print('='*60)
    print('  GPT-2 124M PRISM RECIPE')
    print('='*60)
    t0 = time.time()
    r = subprocess.run([
        'python', 'train.py', 'config/train_gpt2.py',
        f'--max_iters={STEPS}',
        f'--eval_interval={EVAL}',
        '--eval_iters=50',
        '--log_interval=100',
        '--out_dir=out-gpt2-prism',
        '--wandb_log=False',
        '--compile=True',
        '--prism_init=True',
        '--prism_align=0.75',
        f'--prism_spectra={cache}/spectra.json',
        f'--prism_directions={cache}/directions.pt',
        '--prism_mod=0.01',
        '--prism_mod_decay=0.9999',
        # Keep default GPT-2 LR (6e-4) — don't halve it like Shakespeare
        # The LR sweet spot may be different at this scale
        # Single GPU overrides
        '--gradient_accumulation_steps=40',
        '--batch_size=12',
    ], capture_output=True, text=True, timeout=36000)
    wall = time.time() - t0
    with open(log_path, 'w') as f:
        f.write(r.stdout)
    if r.returncode != 0:
        print(f'ERROR: {r.stderr[-500:]}')
    print(f'Wall time: {wall:.0f}s ({wall/3600:.1f}h)')
    for line in r.stdout.split('\n')[-50:]:
        if 'val loss' in line:
            print(f'  {line.strip()}')

In [ ]:
# Cell 5: Compare
import re, json

def parse_val(path):
    val = {}
    with open(path) as f:
        for line in f:
            m = re.search(r'step (\d+):.*val loss ([\d.]+)', line)
            if m: val[int(m.group(1))] = float(m.group(2))
    return val

b = parse_val('/content/gpt2_baseline.txt')
p = parse_val('/content/gpt2_prism.txt')

if not b or not p:
    print('Missing results. Run cells 3 and 4 first.')
else:
    bb = min(b.values())
    bs = min(b, key=b.get)
    pb = min(p.values())
    ps = min(p, key=p.get)
    
    hit = next((s for s in sorted(p) if p[s] <= bb), None)
    score = bs / hit if hit else 0
    
    print('='*60)
    print('  GPT-2 124M — PRISM SCORE')
    print('='*60)
    print(f'  Model: GPT-2 small (124M), OpenWebText')
    print(f'  Teacher: pretrained GPT-2 from HuggingFace (free)')
    print(f'')
    print(f'  Baseline best: {bb:.4f} @ step {bs}')
    print(f'  Prism best:    {pb:.4f} @ step {ps}')
    if hit:
        print(f'  Prism hit baseline @ step {hit}')
        print(f'')
        print(f'  PRISM SCORE: {score:.1f}x')
    else:
        print(f'  Prism never reached baseline quality.')
        print(f'  PRISM SCORE: N/A')
    
    b5 = b.get(max(b.keys()), 0)
    p5 = p.get(max(p.keys()), 0)
    print(f'')
    print(f'  @ final step: baseline={b5:.4f}, prism={p5:.4f}')
    print(f'  Overfitting: baseline={"yes" if b5 > bb*1.05 else "no"}, prism={"yes" if p5 > pb*1.05 else "no"}')
    
    print(f'\n  Convergence:')
    print(f'  {"Step":>8s}  {"Baseline":>10s}  {"Prism":>10s}  {"Winner":>8s}')
    print(f'  {"-"*40}')
    for step in sorted(set(b.keys()) & set(p.keys())):
        bv = b[step]
        pv = p[step]
        w = 'PRISM' if pv < bv else 'BASE'
        print(f'  {step:>8d}  {bv:>10.4f}  {pv:>10.4f}  {w:>8s}')
    
    # Save
    with open('/content/gpt2_prism_score.json', 'w') as f:
        json.dump({
            'model': 'gpt2-124m',
            'dataset': 'openwebtext',
            'prism_score': score,
            'baseline_best': bb, 'baseline_best_step': bs,
            'prism_best': pb, 'prism_best_step': ps,
            'prism_hit_step': hit,
        }, f, indent=2)
    print(f'\n  Saved /content/gpt2_prism_score.json')

In [ ]:
# Cell 6: Download
from google.colab import files
for f in ['/content/gpt2_prism_score.json',
          '/content/gpt2_baseline.txt',
          '/content/gpt2_prism.txt']:
    try: files.download(f)
    except: pass